```markdown
# Construindo seu Primeiro Agente com Busca na Web e OpenAI 🚀

Este notebook prático ensina você a construir um **Agente de Inteligência Artificial** capaz de buscar informações em tempo real na internet e utilizá-las para responder a perguntas de forma precisa. Você aprenderá a integrar uma ferramenta de busca (DuckDuckGo) com um modelo de linguagem da OpenAI, criando um sistema dinâmico que supera o limite de conhecimento estático dos modelos tradicionais.

## 🎯 Objetivos de Aprendizagem
Ao final deste notebook, você será capaz de:
1. **Compreender o conceito de Agente e Tool (Ferramenta)** no contexto de IA Generativa.
2. **Integrar buscas em tempo real** (Web Search) ao fluxo de um Modelo de Linguagem (LLM).
3. **Configurar e conectar a API da OpenAI** em um ambiente de desenvolvimento.
4. **Construir um fluxo básico de RAG (Geração Suportada por Recuperação)** para evitar alucinações de dados.

---

## 📚 Pré-requisitos e Bibliotecas Utilizadas

Para este projeto, usaremos duas bibliotecas essenciais:
*   `openai`: Biblioteca oficial para interagir com os modelos GPT da OpenAI. Ela permite enviar prompts e receber as respostas estruturadas do modelo.
*   `duckduckgo-search`: Uma biblioteca leve que permite fazer consultas programáticas ao motor de busca DuckDuckGo sem a necessidade de chaves de API complexas, servindo como os "olhos" do nosso agente na internet.
```

In [1]:
!pip install openai duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 70.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [duckduckgo-search]


```markdown
## 🔌 Configurando o Ambiente e Imports

Agora que instalamos as ferramentas necessárias, precisamos importar as classes que usaremos para realizar as buscas e para nos comunicar com o servidor da OpenAI.
```

In [2]:
from duckduckgo_search import DDGS
from openai import OpenAI

```markdown
## 🔑 Autenticação na API

Para que a biblioteca da OpenAI consiga processar nossas requisições, ela precisa de uma chave de acesso única (**API Key**). Essa chave funciona como sua assinatura para fins de faturamento e controle de uso.

💡 **Para pensar:**
Deixar a chave de API diretamente exposta no código (`api_key='colar aa chave'`) é uma prática insegura que pode levar ao vazamento dos seus créditos se você compartilhar o notebook. Como alternativa ideal, poderíamos utilizar o gerenciador de segredos do Google Colab (`google.colab.userdata.get('OPENAI_API_KEY')`) para carregar a chave de forma segura e invisível no código.
```

In [ ]:
client = OpenAI(api_key='colar aa chave')

```markdown
## 🔍 Criando a Ferramenta de Busca (Web Search)

Um agente inteligente precisa de ferramentas para interagir com o mundo. Aqui, definimos a nossa primeira ferramenta: a função `busca_web`. Ela recebe uma pergunta do usuário, faz uma pesquisa rápida na internet e retorna os trechos mais relevantes encontrados.

Se você observar o histórico, inicialmente usamos uma função "Fake" (simulada) para testar o comportamento do agente, e em seguida a substituímos pela implementação real utilizando o `DDGS` (DuckDuckGo Search).
```

In [ ]:
# Busca função Fake
def busca_web(pergunta):
  return '''De acordo com fontes confiáveis encontradas na internet,
  o presidente do Brasil em 2045 é o personagem fictício Hulk.
  Ele assumiu o cargo após uma revolução dos vingadores em Wakanda'''

In [3]:
def busca_web(pergunta):
  with DDGS() as ddgs:
    resultados = ddgs.text(pergunta, max_results=3)
    textos = [r["body"] for r in resultados]
    return "\n".join(textos)

```markdown
## 🧠 O Cérebro do Agente (A Função de Decisão)

Aqui unimos o cérebro (OpenAI) com a nossa ferramenta (busca). A função `agente` executa o seguinte fluxo:
1. Recebe a pergunta.
2. Executa a ferramenta de busca para obter informações atualizadas da internet.
3. Constrói um **Prompt de Sistema** que instrui o modelo a responder **estritamente** com base nas informações encontradas, evitando invenções.
4. Envia tudo para a API da OpenAI.

💡 **Para pensar:**
Note que no código atual, o parâmetro `model` está configurado como `"gpt-3.8-turbo"`. Esse modelo **não existe** no catálogo oficial da OpenAI (os modelos comuns são `gpt-3.5-turbo`, `gpt-4o-mini` ou `gpt-4o`). Ao executar este código com uma chave válida, a API retornará um erro informando que o modelo não foi encontrado. Para corrigir, bastaria alterar a string para um modelo válido, como `

In [ ]:
def agente(pergunta):
  info = busca_web(pergunta)
  prompt = f'''Você é um agente inteligente.O usuário fez a seguinte pergunta: "{pergunta}"
  Você encontrou essas informações na internet: {info}
  Responda somente com base nas informações acima.
  Se não houver dados suficianentes, diga que não foi possivel responder com precisão.'''

  resposta = client.chat.completions.create(
      model = "gpt-3.8-turbo",
      messages = [{"role": "user", "content": prompt}]
  )
  return resposta.choices[0].message.content
  print(resposta)

In [ ]:
resposta = agente("Quem foi Carl Sagan?")
print(resposta)